In [3]:
import pandas as pd
import pathlib

ZONE_COMFORT={
    "front_room":21,
    "backroom":19,
    "bedroom_1":19,
    "bedroom_2":19,
    "bedroom_3":12,
    "bathroom":19,
    "hall_downstairs":21,
    "hall_upstairs":12,
    "kitchen":21,
}

# ZONE_COMFORT={
#     "front_room":22,
#     "backroom":20,
#     "bedroom_1":20,
#     "bedroom_2":20,
#     "bedroom_3":12,
#     "bathroom":20,
#     "hall_downstairs":22,
#     "hall_upstairs":12,
#     "kitchen":21,
# }

def generate_heating_schedules(src_dir,dst_dir,cooldown_minutes=30):
    src=pathlib.Path(src_dir)
    dst=pathlib.Path(dst_dir)
    dst.mkdir(exist_ok=True)

    steps_cooldown=cooldown_minutes//10

    for f in src.glob("occupancy_*.sch"):
        df=pd.read_csv(f)
        col=df.columns[0]
        occ=df[col].astype(int).values

        zone=f.stem.replace("occupancy_","")
        comfort=ZONE_COMFORT[zone]
        setback=comfort if comfort==12 else 16

        heat=[setback]*len(occ)
        last_occ=False
        cooldown_left=0

        for i in range(len(occ)):
            if occ[i]>0:
                heat[i]=comfort
                last_occ=True
                cooldown_left=steps_cooldown
            else:
                if last_occ and cooldown_left>0:
                    heat[i]=comfort
                    cooldown_left-=1
                else:
                    heat[i]=setback
                    last_occ=False

        out=pd.DataFrame({col:heat})
        new_name=f"{zone}_heating_schedule.sch"
        out.to_csv(dst/new_name,index=False)


In [4]:
generate_heating_schedules(
    src_dir="/workspaces/CUBES/beizaee/input_data",
    dst_dir="/workspaces/CUBES/beizaee/input_data"
)
